In [ ]:
import json
import time
from getpass import getpass
from pathlib import Path

import pandas as pd
import requests


# Entering the credentials privately when the script runs instead of publicly storing credential string
CLIENT_ID = input("Spotify Client ID: ").strip()
CLIENT_SECRET = getpass("Spotify Client Secret: ").strip()

TRACKS_PER_GENRE = 200
MARKET = "US"

# Slower pacing to reduce api request strain
REQUEST_INTERVAL = 3
MAX_RETRY_WAIT = 300
MAX_RATE_RETRIES = 3

OUTPUT_FILE = Path("spotify_ds2.csv")
CHECKPOINT_FILE = Path("spotify-ds-checkpoint2.json")

GENRES = [
    "pop", "rock", "hip-hop", "rap", "r&b", "soul",
    "jazz", "blues", "classical", "country", "folk",
    "bluegrass", "electronic", "dance", "house",
    "techno", "trance", "dubstep", "ambient", "disco",
    "funk", "metal", "punk", "alternative", "indie",
    "reggae", "reggaeton", "ska", "latin", "salsa",
    "bachata", "bossa nova", "samba", "k-pop", "j-pop",
    "afrobeat", "afrobeats", "gospel", "soundtrack", "lo-fi",
]

COLUMNS = [
    "album_id",
    "album_name",
    "album_type",
    "album_release_date",
    "album_total_tracks",
    "year",
    "query_genres",
    "track_id",
    "track_number",
    "track_name",
    "track_explicit",
    "duration_ms",
    "duration_minutes",
    "artists_names",
    "artists_ids",
    "principal_artist_name",
    "principal_artist_id",
]

In [14]:
class RateLimitReached(RuntimeError):
    pass


class SpotifyClient:
    #Store the credentials create a resusable session
    # initialize timers for token expiration & request pacing
    def __init__(self, client_id, client_secret):
        self.client_id = client_id
        self.client_secret = client_secret
        self.session = requests.Session()
        self.token = None
        self.token_expires_at = 0
        self.last_request_at = None

    # requesting access token / scheduling renewal before expiration
    def renew_token(self):
        response = self.session.post(
            "https://accounts.spotify.com/api/token",
            auth=(self.client_id, self.client_secret),
            data={"grant_type": "client_credentials"},
            timeout=30,
        )
        response.raise_for_status()

        data = response.json()
        self.token = data["access_token"]
        self.token_expires_at = (
            time.monotonic() + data["expires_in"] - 60
        )

        # sending authenticated get request w/ spacing between requests
        # renewing any expired tokens
        # handling any rate limits by waiting or just stopping the collection
    def get(self, path, params=None):
        auth_retried = False
        server_retries = 0
        rate_retries = 0

        while True:
            if (
                self.token is None
                or time.monotonic() >= self.token_expires_at
            ):
                self.renew_token()

            if self.last_request_at is not None:
                elapsed = time.monotonic() - self.last_request_at
                time.sleep(max(0, REQUEST_INTERVAL - elapsed))

            self.last_request_at = time.monotonic()

            response = self.session.get(
                f"https://api.spotify.com/v1{path}",
                headers={
                    "Authorization": f"Bearer {self.token}"
                },
                params=params,
                timeout=30,
            )

            if response.status_code == 429:
                try:
                    wait_seconds = max(
                        1,
                        float(response.headers.get("Retry-After", 60)),
                    )
                except (TypeError, ValueError):
                    wait_seconds = 60

                if (
                    wait_seconds > MAX_RETRY_WAIT
                    or rate_retries >= MAX_RATE_RETRIES
                ):
                    raise RateLimitReached(
                        "Spotify rate limit reached. "
                        f"Requested wait: {wait_seconds / 3600:.2f} hours. "
                        "Stop collection and resume after that period."
                    )

                rate_retries += 1
                print(
                    f"Rate limited. Waiting {wait_seconds + 1:.0f} "
                    "seconds...",
                    flush=True,
                )
                time.sleep(wait_seconds + 1)
                continue

            if response.status_code == 401 and not auth_retried:
                self.renew_token()
                auth_retried = True
                continue

            if response.status_code in (500, 502, 503, 504):
                if server_retries < 4:
                    time.sleep(2 ** (server_retries + 1))
                    server_retries += 1
                    continue

            response.raise_for_status()
            return response.json()
    # close and release connections
    def close(self):
        self.session.close()

# converting a track search result into a dataset
# extract info for album & artist, duration, and genre labels
def build_row(track, query_genres):
    """Build a row directly from search results; no extra API calls."""
    album = track.get("album") or {}
    artists = track.get("artists") or []
    principal = artists[0] if artists else {}

    release_date = album.get("release_date")
    duration_ms = track.get("duration_ms")

    year = None
    if release_date and release_date[:4].isdigit():
        year = int(release_date[:4])

    return {
        "track_id": track.get("id"),
        "track_name": track.get("name"),
        "album_id": album.get("id"),
        "album_name": album.get("name"),
        "year": year,
        "query_genres": sorted(set(query_genres)),
        "album_type": album.get("album_type"),
        "album_total_tracks": album.get("total_tracks"),
        "album_release_date": release_date,
        "track_explicit": track.get("explicit"),
        "track_number": track.get("track_number"),
        "duration_ms": duration_ms,
        "duration_minutes": (
            round(duration_ms / 60000, 3)
            if duration_ms is not None
            else None
        ),
        "artists_names": [artist.get("name") for artist in artists],
        "artists_ids": [artist.get("id") for artist in artists],
        "principal_artist_name": principal.get("name"),
        "principal_artist_id": principal.get("id"),
    }

# loading saved tracks from the checkpoint (if it exists)
# if theres none make an empty collection
def load_progress():
    if CHECKPOINT_FILE.exists():
        with CHECKPOINT_FILE.open("r", encoding="utf-8") as file:
            state = json.load(file)
    else:
        state = {}

    state.setdefault("tracks", {})

    # Separate progress from the old collector so existing tracks
    # can acquire query labels without deleting previous data
    state.setdefault("search_only_offsets", {})
    state.setdefault("search_only_completed", [])

    # Preserve old checkpoint fields and data
    for row in state["tracks"].values():
        row.setdefault("query_genres", [])

        duration_ms = row.get("duration_ms")
        if duration_ms is not None:
            row["duration_minutes"] = round(duration_ms / 60000, 3)

    return state

# save progress to json checkpoint 
# export selected columns to csv 
# temporary files written first then replace save files
def save_progress(state):
    checkpoint_temp = CHECKPOINT_FILE.with_suffix(".json.tmp")

    with checkpoint_temp.open("w", encoding="utf-8") as file:
        json.dump(state, file, ensure_ascii=False)

    checkpoint_temp.replace(CHECKPOINT_FILE)

    df = pd.DataFrame(
        list(state["tracks"].values()),
        columns=COLUMNS,
    )

    for column in ["query_genres", "artists_names", "artists_ids"]:
        df[column] = df[column].map(
            lambda value: (
                json.dumps(value, ensure_ascii=False)
                if isinstance(value, list)
                else value
            )
        )

    csv_temp = OUTPUT_FILE.with_suffix(".csv.tmp")
    df.to_csv(csv_temp, index=False, encoding="utf-8")
    csv_temp.replace(OUTPUT_FILE)


In [15]:
# validate settings and collect tracks 
# handle limit stops and keyboard interruptions then save progress
def main():
    if not CLIENT_ID or not CLIENT_SECRET:
        raise ValueError("Enter your Spotify credentials.")

    if not 1 <= TRACKS_PER_GENRE <= 1000:
        raise ValueError("TRACKS_PER_GENRE must be between 1 and 1000.")

    state = load_progress()
    client = SpotifyClient(CLIENT_ID, CLIENT_SECRET)

    try:
        for genre in GENRES:
            query = f'genre:"{genre}"'
            progress_key = f"{query}|{MARKET}|{TRACKS_PER_GENRE}"

            if progress_key in state["search_only_completed"]:
                print(f"Already completed: {genre}")
                continue

            offset = state["search_only_offsets"].get(
                progress_key, 0
            )
            print(f"Collecting: {genre}, starting at {offset}")

            while offset < TRACKS_PER_GENRE:
                page_size = min(10, TRACKS_PER_GENRE - offset)

                result = client.get(
                    "/search",
                    params={
                        "q": query,
                        "type": "track",
                        "market": MARKET,
                        "limit": page_size,
                        "offset": offset,
                    },
                )

                page = result["tracks"]
                tracks = page.get("items", [])

                if not tracks:
                    break

                for track in tracks:
                    if not track or not track.get("id"):
                        continue

                    track_id = track["id"]
                    existing = state["tracks"].get(track_id, {})
                    query_genres = existing.get("query_genres") or []

                    # Preserve every search label for duplicate tracks.
                    row = build_row(track, query_genres + [genre])
                    existing.update(row)
                    state["tracks"][track_id] = existing

                offset += len(tracks)
                state["search_only_offsets"][progress_key] = offset
                save_progress(state)

                print(
                    f"  {genre}: {offset} results processed; "
                    f"{len(state['tracks'])} unique tracks saved."
                )

                if not page.get("next"):
                    break

            state["search_only_completed"].append(progress_key)
            save_progress(state)

    except RateLimitReached as error:
        print(f"\n{error}")

    except KeyboardInterrupt:
        print("\nCollection stopped by you.")

#close the client when collection stops
    finally:
        client.close()
        save_progress(state)
        print(
            f"\nSaved {len(state['tracks'])} unique tracks "
            f"to {OUTPUT_FILE}"
        )
        print(f"Resume information saved to {CHECKPOINT_FILE}")


if __name__ == "__main__":
    main()

Collecting: pop, starting at 0
  pop: 10 results processed; 10 unique tracks saved.
  pop: 20 results processed; 20 unique tracks saved.
  pop: 30 results processed; 30 unique tracks saved.
  pop: 40 results processed; 40 unique tracks saved.
  pop: 50 results processed; 50 unique tracks saved.
  pop: 60 results processed; 60 unique tracks saved.
  pop: 70 results processed; 70 unique tracks saved.
  pop: 80 results processed; 80 unique tracks saved.
  pop: 90 results processed; 90 unique tracks saved.
  pop: 100 results processed; 100 unique tracks saved.
Collecting: rock, starting at 0
  rock: 10 results processed; 108 unique tracks saved.
  rock: 20 results processed; 118 unique tracks saved.
  rock: 30 results processed; 127 unique tracks saved.
  rock: 40 results processed; 134 unique tracks saved.
  rock: 50 results processed; 144 unique tracks saved.
  rock: 60 results processed; 154 unique tracks saved.
  rock: 70 results processed; 164 unique tracks saved.
  rock: 80 results p